In [9]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [10]:
# 1. Initialize data and tokenizer in the main process
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
raw_datasets = load_dataset("imdb")

In [11]:
# 2. Update the function signature to accept the tokenizer explicitly
def tokenize_function(examples, tokenizer_object):
    # Notice we use tokenizer_object, which is safely passed to the workers
    return tokenizer_object(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=128
    )

print("Starting parallel dataset tokenization...")

Starting parallel dataset tokenization...


In [12]:
# 3. Use fn_kwargs to pass the tokenizer into the workers safely
tokenized_datasets = raw_datasets.map(
    tokenize_function, 
    batched=True, 
    num_proc=2,
    fn_kwargs={"tokenizer_object": tokenizer} # Maps to the argument above
)

print("\nPost-Tokenization Dataset Structure:")
print(tokenized_datasets["train"])

Map (num_proc=2): 100%|██████████████████████████████████████████████████| 50000/50000 [00:50<00:00, 995.55 examples/s]


Post-Tokenization Dataset Structure:
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})
